## Decision Tree from Stratch

#### Libraries

In [88]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

#### Load Case 1

In [89]:
df_train1 = pd.read_csv('../data/case1/train1.csv')
df_test1 = pd.read_csv('../data/case1/test1.csv')

In [90]:
X_train = df_train1.drop(columns = ['flow_m3_h' , 'pump_id' , 'operating_state'])
y_train = df_train1['flow_m3_h']

X_test = df_test1.drop(columns = ['flow_m3_h' , 'pump_id' , 'operating_state'])
y_test = df_test1['flow_m3_h']

In [91]:
def gini_impurity(labels):
    '''
    compute Gini Impurity for a set of labels
    formula: gini = 1 - sum(p_1^2), where p_1 is proportion of class 1
    ''' 
    if len(labels) == 0:
        return 0

    # count occurences of each class
    unique_classes , counts = np.unique(labels , return_counts = True)
    proportions = counts / len(labels)

    # compute Gini Impurity
    gini = 1 - np.sum(proportions ** 2)
    return gini

def weighted_impurity(left_labels , right_labels , impurity_func = gini_impurity):
    ''' 
    compute weighted impurity for a split
    formula: weighted impurity = (n_left / n_total) * impurity_left + (n_right / n_total) * impurity_right
    ''' 
    n_left = len(left_labels)
    n_right = len(right_labels)
    n_total = n_left + n_right

    if n_total == 0:
        return 0

    # compute weighted average of impurities
    weighted_imp = (n_left / n_total) * impurity_func(left_labels) + (n_right / n_total) * impurity_func(right_labels)
    return weighted_imp

In [92]:
# reference: exercise session ...

class Node:
    '''
    A node in the decision tree
    '''
    def __init__(self , feature = None , threshold = None , left = None , right = None , value = None):
        self.feature = feature          # index of feature to split on
        self.threshold = threshold      # treshold value for the split
        self.left = left                # left child node
        self.right = right              # right child node
        self.value = value              # class labels for leaf nodes

    def is_leaf(self):
        '''
        check if node is a leaf
        '''
        return self.value is not None
    
class DecisionTreeFromScratch:
    '''
    a decision tree classifier from scratch
    ''' 
    def __init__(self , max_depth = None , min_samples_split = 2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self , X , y):
        ''' 
        build the decision tree
        ''' 
        # because root exepct NumPy array
        X = np.array(X)
        y = np.array(y)

        self.root = self._build_tree(X , y , depth = 0)
        return self
    
    def _build_tree(self , X , y , depth):
        ''' 
        recursively build the decision tree
        ''' 
        n_samples , n_features = X.shape
        n_classes = len(np.unique(y))

        # stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) or \
            n_classes == 1 or \
            n_samples < self.min_samples_split:
                #create a leaf node with the most common class
                leaf_value = self._most_common_label(y)
                return Node(value = leaf_value)

        # find best split
        best_feature , best_threshold = self._best_split(X , y)

        # if not split improves tree, create a leaf node
        if best_feature is None:
            leaf_value = self._most_common_label(y)
            return Node(value = leaf_value)

        # split the dataset
        left_indices = X[: , best_feature] <= best_threshold
        right_indices = X[: , best_feature] > best_threshold

        # recursively build left and right subtrees
        left_child = self._build_tree(X[left_indices] , y[left_indices] , depth = 1)
        right_child = self._build_tree(X[right_indices] , y[right_indices] , depth = 1)

        return Node(
            feature = best_feature,
            threshold = best_threshold,
            left = left_child,
            right = right_child
        )
    
    def _best_split(self , X , y):
        ''' 
        find the best split for the dataset
        '''
        n_samples , n_features = X.shape

        if n_samples <= 1:
            return None , None

        best_gini = float('inf')
        best_feature = None
        best_threshold = None

        # try all features
        for feature_idx in range(n_features):
            X_col = X[: , feature_idx]
            thresholds = np.unique(X_col)

            # try all possible thresholds
            for i in range(len(thresholds) - 1):
                threshold = (thresholds[i] + thresholds[i + 1]) / 2

                # split the data
                left_indices = X_col <= threshold
                right_indices = X_col > threshold

                left_y = y[left_indices]
                right_y = y[right_indices]

                # compute weigthed Gini Impurity
                w_gini = weighted_impurity(left_y , right_y , gini_impurity)

                # update best split if better
                if w_gini < best_gini:
                    best_gini = w_gini
                    best_feature = feature_idx
                    best_threshold = threshold

        return best_feature , best_threshold

    def _most_common_label(self , y):
        ''' 
        return the most common label in y
        '''
        unique, counts = np.unique(y , return_counts = True)
        return unique[np.argmax(counts)]

    def predict(self , X):
        ''' 
        predict class labels for samples in X
        ''' 
        X = np.asarray(X)
        return np.array([self._traverse_tree(x , self.root) for x in X])
    
    def _traverse_tree(self , x , node):
        ''' 
        traverse the trees to make a prediction for single sample
        ''' 
        # if we are at leaf node, return its value
        if node.is_leaf():
            return node.value

        # otherwise, traverse left or right based on feature value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x , node.left)
        else:
            return self._traverse_tree(x , node.right)



In [104]:
for depth in [1, 2, 3, 5, 7, 10]:
    for splits in [2, 4, 8]:
        dt = DecisionTreeFromScratch(
            max_depth = depth,
            min_samples_split = splits
        )

        dt.fit(X_train, y_train)

        y_train_pred = dt.predict(X_train)
        y_test_pred = dt.predict(X_test)

        train_mse = mean_squared_error(y_train , y_train_pred)
        test_mse = mean_squared_error(y_test , y_test_pred)

        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)

        print(
            f"Depth = {depth} and, "
            f"Minimum samples = {splits}: "
            f"Train MSE = {train_mse:.3f}, "
            f"Test MSE = {test_mse:.3f}, "
            f"Train R2 = {train_r2:.3f}, "
            f"Test R2 = {test_r2:.3f}"
        )

Depth = 1 and, Minimum samples = 2: Train MSE = 510.099, Test MSE = 498.600, Train R2 = -1.994, Test R2 = -2.421
Depth = 1 and, Minimum samples = 4: Train MSE = 510.099, Test MSE = 498.600, Train R2 = -1.994, Test R2 = -2.421
Depth = 1 and, Minimum samples = 8: Train MSE = 510.099, Test MSE = 498.600, Train R2 = -1.994, Test R2 = -2.421
Depth = 2 and, Minimum samples = 2: Train MSE = 0.000, Test MSE = 324.719, Train R2 = 1.000, Test R2 = -1.228
Depth = 2 and, Minimum samples = 4: Train MSE = 101.198, Test MSE = 333.007, Train R2 = 0.406, Test R2 = -1.285
Depth = 2 and, Minimum samples = 8: Train MSE = 220.950, Test MSE = 375.407, Train R2 = -0.297, Test R2 = -1.576
Depth = 3 and, Minimum samples = 2: Train MSE = 0.000, Test MSE = 324.719, Train R2 = 1.000, Test R2 = -1.228
Depth = 3 and, Minimum samples = 4: Train MSE = 101.198, Test MSE = 333.007, Train R2 = 0.406, Test R2 = -1.285
Depth = 3 and, Minimum samples = 8: Train MSE = 220.950, Test MSE = 375.407, Train R2 = -0.297, Test R2 